# Intelligent Predictive Maintenance System (AI4I 2020)
## Exploratory Data Analysis & Integrity Audit (Stage 1)

**Course Project:** Advanced Machine Learning for Industrial Equipment Maintenance  
**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI Machine Learning Repository)  
**Author:** ML Engineering Team  
**Environment:** Python 3.13 | pandas, scikit-learn, pyGAM, SHAP, XGBoost

---
### Objectives of this Notebook
1. **Data Integrity Audit:** Verify zero missing values, zero duplicates, and confirm data types.
2. **Target Imbalance Analysis:** Quantify binary class prevalence of `Machine failure` (~3.39%).
3. **Failure Mode Deconstruction:** Analyze the 5 failure modes (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`), their co-occurrence, and document the 9 unlabeled failure cases and 18 non-failure flagged anomalies.
4. **Sensor Distributions & Physics Regimes:** Examine temperature, speed, torque, and wear across product types and failure classes.
5. **Outlier Detection:** Apply mathematical IQR and Z-score criteria, documenting why valid extreme sensor states must be preserved.
6. **Correlation & Bivariate Structure:** Identify collinearity and physically-motivated feature interactions.


In [1]:
import os
import sys
from pathlib import Path

# Set headless matplotlib and config dir
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from scipy import stats

# Add project root to sys.path
for candidate in [Path(".").resolve(), Path("..").resolve(), Path(__file__).resolve().parent.parent if "__file__" in globals() else Path(".")]:
    if (candidate / "config.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    PROJECT_ROOT = Path(".").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import RAW_DATA_PATH, VIZ_DIR
print("Libraries imported successfully. Dataset path:", RAW_DATA_PATH)


Libraries imported successfully. Dataset path: /Users/surya/Downloads/human+activity+recognition+using+smartphones/predictive-maintenance-system/data/ai4i2020.csv


### 1. Load Raw Data & Check Integrity
The raw dataset contains 10,000 synthetic records reflecting real industrial milling machine sensor streams.


In [2]:
df = pd.read_csv(RAW_DATA_PATH)
print("Dataset Shape:", df.shape)
print("Missing values total:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df.head()


Dataset Shape: (10000, 14)
Missing values total: 0
Duplicate rows: 0


### 2. Target Class Imbalance (`Machine failure`)
In predictive maintenance, machine failures are rare events. High accuracy can be achieved trivially by predicting all non-failures (96.61% accuracy for a null model). Hence, **PR-AUC** and **Recall** at calibrated decision thresholds are the primary evaluation criteria.


In [3]:
target_counts = df['Machine failure'].value_counts()
target_pct = df['Machine failure'].value_counts(normalize=True) * 100
imbalance_ratio = target_counts[0] / target_counts[1]

print(f"Normal (0):  {target_counts[0]} ({target_pct[0]:.2f}%)")
print(f"Failure (1): {target_counts[1]} ({target_pct[1]:.2f}%)")
print(f"Imbalance ratio: {imbalance_ratio:.2f} : 1")


Normal (0):  9661 (96.61%)
Failure (1): 339 (3.39%)
Imbalance ratio: 28.50 : 1


### 3. Product Type Breakdown & Stratification
Products are divided into Low (L, 60%), Medium (M, 30%), and High (H, 10%) quality variants, which dictate machine load thresholds and failure propensities.


In [4]:
print(df['Type'].value_counts())
type_fail = df.groupby('Type')['Machine failure'].agg(['count', 'sum', 'mean'])
type_fail['failure_rate_%'] = type_fail['mean'] * 100
print("\nFailure Rate by Type:")
print(type_fail)


Type
L    6000
M    2997
H    1003
Name: count, dtype: int64

Failure Rate by Type:
      count  sum      mean  failure_rate_%
Type                                      
H      1003   21  0.020937        2.093719
L      6000  235  0.039167        3.916667
M      2997   83  0.027694        2.769436


### 4. Failure Mode Flags & Anomalous Row Analysis
The dataset includes five failure mode indicators:
- **TWF:** Tool Wear Failure (tool wear reaches replacement threshold ~200-240 min)
- **HDF:** Heat Dissipation Failure (temperature difference < 8.6 K and speed < 1380 rpm)
- **PWF:** Power Failure (mechanical power < 3500 W or > 9000 W)
- **OSF:** Overstrain Failure (product of tool wear and torque exceeds variant limit)
- **RNF:** Random Failure (independent 0.1% random failure process)

#### Overlap Breakdown:
1. **Anomaly A:** 9 rows have `Machine failure == 1` but all 5 failure mode flags are 0.
2. **Anomaly B:** 18 rows have `Machine failure == 0` but a failure mode flag is 1 (all 18 are `RNF == 1`).


In [5]:
flags = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
print("Flag totals across all 10,000 rows:")
print(df[flags].sum())

failed = df[df['Machine failure'] == 1]
flag_sums = failed[flags].sum(axis=1)

print("\nFailed machines (N=339) flag overlap:")
print("  - 0 flags set (unlabeled failure):", (flag_sums == 0).sum())
print("  - 1 flag set:                     ", (flag_sums == 1).sum())
print("  - >=2 flags set (cascading failure):", (flag_sums >= 2).sum())

non_failed = df[df['Machine failure'] == 0]
flagged_non_failed = non_failed[non_failed[flags].sum(axis=1) > 0]
print("\nNon-failed machines (N=9661) with flags > 0:")
print("  - Total anomalous rows:", len(flagged_non_failed))
print(flagged_non_failed[flags].sum())


Flag totals across all 10,000 rows:
TWF     46
HDF    115
PWF     95
OSF     98
RNF     19
dtype: int64

Failed machines (N=339) flag overlap:
  - 0 flags set (unlabeled failure): 9
  - 1 flag set:                      306
  - >=2 flags set (cascading failure): 24

Non-failed machines (N=9661) with flags > 0:
  - Total anomalous rows: 18
TWF     0
HDF     0
PWF     0
OSF     0
RNF    18
dtype: int64


### 5. Outlier Detection: IQR & Z-Score Analysis
We compute outlier counts using two standard mathematical definitions:
1. **IQR Rule:** Values outside $[Q_1 - 1.5\cdot\text{IQR}, Q_3 + 1.5\cdot\text{IQR}]$.
2. **Z-Score Rule:** Values with $|z| = \left|\frac{x - \mu}{\sigma}\right| > 3$.

**ML Engineering Decision:** We do **NOT** remove these outliers. In industrial predictive maintenance, extreme sensor readings (e.g. RPM > 2800 or Torque > 70 Nm) correspond to physical overstrain and power failure regimes. Removing them would eliminate critical failure signals!


In [6]:
sensor_cols = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

outlier_records = []
for c in sensor_cols:
    s = df[c]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    iqr_out = s[(s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)]
    z_out = s[np.abs(stats.zscore(s)) > 3]
    outlier_records.append({
        'Feature': c,
        'Mean': round(s.mean(), 2),
        'Std': round(s.std(), 2),
        'IQR Outliers': len(iqr_out),
        'IQR %': round(len(iqr_out)/len(s)*100, 2),
        'Z (|z|>3) Outliers': len(z_out),
        'Z %': round(len(z_out)/len(s)*100, 2)
    })

outlier_df = pd.DataFrame(outlier_records)
print(outlier_df.to_string(index=False))


                Feature    Mean    Std  IQR Outliers  IQR %  Z (|z|>3) Outliers  Z %
    Air temperature [K]  300.00   2.00             0   0.00                   0 0.00
Process temperature [K]  310.01   1.48             0   0.00                   0 0.00
 Rotational speed [rpm] 1538.78 179.28           418   4.18                 164 1.64
            Torque [Nm]   39.99   9.97            69   0.69                  25 0.25
        Tool wear [min]  107.95  63.65             0   0.00                   0 0.00


### 6. Tool Wear Degradation vs. Failure Risk
As tool wear accumulates, failure probability accelerates dramatically, particularly beyond 200 minutes of continuous operation.


In [7]:
df['wear_bin'] = pd.cut(df['Tool wear [min]'], bins=[0, 50, 100, 150, 200, 250, 300],
                            labels=['0-50m', '51-100m', '101-150m', '151-200m', '201-250m', '251+m'],
                            include_lowest=True)
wear_rates = df.groupby('wear_bin', observed=True)['Machine failure'].agg(['count', 'sum', 'mean'])
wear_rates['failure_rate_%'] = (wear_rates['mean'] * 100).round(2)
print(wear_rates)


          count  sum      mean  failure_rate_%
wear_bin                                      
0-50m      2396   52  0.021703            2.17
51-100m    2271   53  0.023338            2.33
101-150m   2295   50  0.021786            2.18
151-200m   2276   66  0.028998            2.90
201-250m    760  116  0.152632           15.26
251+m         2    2  1.000000          100.00


### 7. Correlation Analysis
Correlation between sensors and failures. Notice the strong negative correlation between Rotational Speed and Torque ($P = \tau \cdot \omega$), and the strong correlation between Air Temperature and Process Temperature.


In [8]:
corr_matrix = df[sensor_cols + ['Machine failure'] + flags].corr()
print("Correlation with Machine failure:")
print(corr_matrix['Machine failure'].sort_values(ascending=False))


Correlation with Machine failure:
Machine failure            1.000000
HDF                        0.575800
OSF                        0.531083
PWF                        0.522812
TWF                        0.362904
Torque [Nm]                0.191321
Tool wear [min]            0.105448
Air temperature [K]        0.082556
Process temperature [K]    0.035946
RNF                        0.004516
Rotational speed [rpm]    -0.044188
Name: Machine failure, dtype: float64


### 8. Key Findings & Implications for Modeling
1. **Target Imbalance:** Positive class is 3.39% (339 / 10,000). Models must use `class_weight='balanced'` or `scale_pos_weight ≈ 28.5`, with PR-AUC as the primary validation metric.
2. **Strict Leakage Guard:** `UDI`, `Product ID`, and the 5 failure mode flags (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) must be strictly excluded from binary machine failure feature matrices.
3. **Physical Interactions:** Strong physical relationships dictate failure:
   - Heat dissipation failure depends on $(T_{\text{process}} - T_{\text{air}})$ and RPM.
   - Power failure depends on mechanical power $P = \tau \cdot \omega$.
   - Overstrain failure depends on $\tau \cdot \text{Tool wear}$.
   Engineering these features in Stage 2 directly reflects the underlying failure mechanics.
4. **Outlier Retention:** Rotational speed (4.18% IQR outliers) and Torque (0.69% IQR outliers) reflect true anomalous operating conditions leading to failure. They must not be truncated or removed.
5. **Multi-label Target Nuances:** The 9 unlabeled failures and 18 flagged non-failures must be explicitly documented and handled in Stage 5.
